In [3]:
"""Inserts"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Any, Callable


### Test

In [ ]:
"""Reading data"""
training_data=pd.read_parquet("data_parquet_2026/train.parquet")
sensors = pd.read_parquet("data_parquet_2026/sensors.parquet")
test_data = pd.read_parquet("data_parquet_2026/test.parquet")

In [ ]:
""" Ségrégation des profiles"""
n=int(len(training_data["power"])/len(training_data["sensor"].unique())/3) #27384/3 = 9128
power1 = training_data.query("sensor =='N0000'")
power2 = training_data.query("sensor =='N0000'")
power3 = training_data.query("sensor =='N0000'")

for senso in training_data["sensor"].unique():
    power1 = pd.concat((power1, training_data[training_data["sensor"] == senso][:n]))
    power2 = pd.concat((power2, training_data[training_data["sensor"] == senso][n+1:2*n]))
    power3 = pd.concat((power3, training_data[training_data["sensor"] == senso][2*n+1:]))

#power1.plot.scatter(x="time", y="power", alpha=0.5)
#power2.plot.scatter(x="time", y="power", alpha=0.5)
#power3.plot.scatter(x="time", y="power", alpha=0.5)

In [ ]:
training_data.info()
#training_data.head(5)
training_data["sensor"]
sensors["sensor"]
#test_data.sample(15)
#sensors.info()
#sensors.query("coor_z == 0") #tous =0
#sensors.head(21)




In [ ]:
#for sens in training_data["sensors"]:
 #   print(sens)
sampleP=training_data.query("time <= 1800000 and sensor=='N418'")
sampleP.head(10)

sampleP.plot.scatter(x="time", y="temperature", alpha=0.5)
#mean.plot.scatter(x="time", y="temperature", alpha=0.5)

In [ ]:
"""graphes tous pour 1 senseur"""

sampleN418 = power1.query(" sensor=='N418'")
#sampleN418.plot.scatter(x="time", y="power", alpha=0.5)
sampleN418.plot.scatter(x="time", y="temperature", alpha=0.5)
sensors.query("sensor == 'N418'")
sampleN418.head()

In [ ]:
b = training_data.query("time <= 10**9 and time >= 0.8*10**9")
b.plot.scatter(x="time", y="power", alpha=0.5)
b.plot.scatter(x="time", y="temperature", alpha=0.5)



In [ ]:
#for sens in training_data["sensors"]:
 #   print(sens)
sampleP=training_data.query("time <= 1800000 and sensor=='N418'")
sampleP.head(10)

sampleP.plot.scatter(x="power", y="temperature", alpha=0.5)
#mean.plot.scatter(x="time", y="temperature", alpha=0.5)

In [ ]:
a = sampleP["power"].diff(periods=864000)

a.head(20)
power1 = training_data.query("power == 500.0000 or power == 1000.0000 or power == 1500.0000")
power1.plot.scatter(x="time", y="power", alpha=0.5)


In [ ]:
"""graphes tous pour 1 senseur"""

sampleN418 = training_data.query(" sensor=='N418'")
sampleN418.plot.scatter(x="time", y="power", alpha=0.5)
sampleN418.plot.scatter(x="power", y="temperature", alpha=0.5)
sensors.query("sensor == 'N418'")

In [ ]:
b = training_data.query("time <= 10**9 and time >= 0.8*10**9")
b.plot.scatter(x="time", y="power", alpha=0.5)
b.plot.scatter(x="time", y="temperature", alpha=0.5)
c=b["power"].pct_change(periods=1)


In [ ]:
sampleN418b = sampleN418.sample(1000)
sampleN418b.plot.scatter(x="time", y="power", alpha=0.5)
sampleN418b.plot.scatter(x="time", y="temperature", alpha=0.5)


## I. First Model: KNN Method

### 1. Distance Metrics A COMPLETER

In [23]:
def man_dist(sample_coo: pd.DataFrame, train_set_coo: pd.DataFrame) -> pd.DataFrame:

    """Computes the Manhattan distance between a sample and all train sensors, and add
    their values in a new column of train_set_coo.
    Args:
        sample_coo: Dataset of the sample's coordinates
        train_set_coo: Dataset of the sensors' coordinates
    Returns:
        train_set_coo_with_dist: train_set_coo with a column "distance", filled.
    """

    return
   


def eucli_dist(sample_coo: pd.DataFrame, train_set_coo: pd.DataFrame) -> pd.DataFrame:

    """Computes the Euclidian distance between a sample and all train sensors, and add
    their values in a new column of train_set_coo.
    Args:
        sample_coo: Dataset of the sample's coordinates
        train_set_coo: Dataset of the sensors' coordinates
    Returns:
        train_set_coo_with_dist: train_set_coo with a column "distance", filled.
    """
   
    return

### 2. Distance Score

In [ ]:
def distance_score(neighbors:pd.DataFrame,
                   score_parameter: float = 1) -> pd.DataFrame:
    
    """Replace the "distance" column by a "score" one to each given neighbor sensor,
    depending on their distance to the sample.
    The closer is the sample, the higher the score. We set an upper limit to it
    to avoid giving "blind confidence" to "very close" sensors.
    Args:
        neighbors: neighbor sensors, with "distance" column, regarding the sample.
        score_parameter: Hyper-parameter, set to 1 by default.
    Returns:
        neighbors: Updated with the "score" column.
    """

    scores = 1 / (score_parameter * neighbors["distance"] + 1)

    # We replace the distance column (which we don't need anymore) by the "score" one.
    neighbors["score"] = scores
    neighbors.drop(columns=["distance"])
    
    return neighbors


### 3. Missing Values manager A COMPLETER (docu)

In [ ]:
def managing_missing_val(sens : pd.DataFrame):
    
    """
    """
    
    sens = sens.reset_index(drop=True)
    temp = 0
    memo = []

    for i, val in enumerate(sens["temperature"]):
        
        if not val == val: # if nan
            memo.append(i)       
                     
        else: # c'est pas un nan
            if temp is not int: # si on a une valeur en mémoire
                if len(memo) != 0: # et qu'il y avait des nan avant
                    sens.loc[memo,"temperature"] = (temp+sens["temperature"][i]) / 2 # on remplace les nan par la moyenne de la mémoire et l'actuel
                    memo = [] # on oublie
                temp = sens["temperature"][i]
                
            else: # pas de valeur en mémoire
                if len(memo) != 0: # mais des nan avant
                    sens.loc[memo,"temperature"]=sens["temperature"][i] # on remplaces les nan par la valeur d'après
                    memo = [] # on oublie
        
    return sens

### 4. K-Nearest Neighbors A COMPLETER

In [ ]:
def find_nearest_neighbors(
    sample_coo: pd.DataFrame, 
    train_set_coo: pd.DataFrame, 
    distance_fn: Callable = eucli_dist, 
    k: int = 7) -> pd.DataFrame:
    
    """Finds the names and distances of the k-Nearest Neighbors of one sample.
    Args:
        sample_coo: Coordinates of the sample
        train_set_coo: Dataset of the the sensors' coordinates
        distance_fn: Distance function
        k: Number of nearest neighbors considered, set to 7 by default
    Returns:
        neighbors: dataframe of the k-nearest neighbors of the sample, with their distances
        stored in a new "distance" column.
    """

    distances = distance_fn(sample_coo, train_set_coo)
    ...
    neighbors = ...

    
    return neighbors

### 5. Prediction Functions

In [ ]:
def prediction_single(
    sample_coo: pd.DataFrame, 
    train_set_coo: pd.DataFrame,
    train_set_temp_at_t: pd.DataFrame,
    distance_fn: Callable = eucli_dist, 
    k: int = 7) -> float:

    """Gives the final prediction of a given sample's temperature, while calling KNN. Calculates the
    mean value of the temperatures of the neighbors, weighted by their distance scores.
    Args:
        sample_coo: Coordinates of the sample
        train_set_coo: Dataset of the sensors' coordinates
        train_set_temp_at_t: Dataset of the sensors' temperatures at the time of the sample
        distance_fn: Distance function
        k: Number of nearest neighbors taken, set to 7 by default
    Returns:
        pred: The value of the predicted temperature of the sample
    """

    global neighbors_dict
    sample_name = sample_coo["sensor"]

    # We check if the current sample was already analyzed in the past
    if sample_name not in neighbors_dict:
        
        # Run KNN
        neighbors = find_nearest_neighbors(sample_coo = sample_coo, dataset = train_set_coo, distance_fn = distance_fn, k = k)

        # The dictionnary is filled directly with the scores (not the distances)
        neighbors_dict[sample_name] = distance_score(neighbors = neighbors)["score"].to_numpy

    temperatures = train_set_temp_at_t[train_set_temp_at_t.sensor in neighbors_dict[sample_name]]["temperature"].to_numpy

    pred = np.sum(neighbors_dict[sample_name] * temperatures) / np.sum(neighbors_dict[sample_name])

    return pred



def prediction(
    validation_set: pd.DataFrame,
    train_set_coo: pd.DataFrame,
    train_set_values: pd.DataFrame,
    distance_fn: Callable = eucli_dist, 
    k: int = 7) -> pd.DataFrame:

    """Gives the final prediction of a given sample's temperature, while calling KNN. Calculates the
    mean value of the temperatures of the neighbors, weighted by their distance scores.
    Args:
        validation_set: Dataframe with names, coordinates, and some values of time of some sensors whom
        we "hide" the temperature values.
        train_set_coo: Dataset of the "train" sensors' coordinates
        train_set_values: Dataset of the "train" sensors' values of time and temperatures.
        distance_fn: Distance function (among Manhattan or Euclidian)
        k: Number of nearest neighbors taken, set to 7 by default
    Returns:
        pred: The validation_set with a column "prediction", filled with the outputs of prediction_single.
    """

    n = validation_set.shape[0]

    # Add new column
    validation_set["prediction"] = np.empty(n)

    # Loop over all the validation set
    for index, sample in validation_set.iterrows():
        time = sample["time"]
        sample["prediction"] = prediction_single(sample_coo = sample.drop(["time","prediction"]),
                                                 train_set_coo = train_set_coo,
                                                 train_set_temp_at_t = train_set_values[("time" == time)],
                                                 distance_fn = distance_fn,
                                                 k = k)
    
    return validation_set



    

### 6. Loss Function A COMPLETER

### Application A COMPLETER

In [4]:
# Reading data
training_data=pd.read_parquet("data_parquet_2026/train.parquet")
sensors = pd.read_parquet("data_parquet_2026/sensors.parquet").drop(columns=["coor_z"])
test_data = pd.read_parquet("data_parquet_2026/test.parquet")

In [ ]:
# Partition of the dataset into "training" and "validation" sets
validation_sensors_coo = sensors.sample(frac =1/5).sort_index()
train_sensors_coo = sensors.drop(validation_sensors_coo.index)

# Initialisation of neighbors_dict, which will stock the rows of the sensors with their scores
# to each given sample already evaluated.
neighbors_dict = {}



    sensor     coor_x  coor_y
105   N104  44.278809     3.5


## II. A few improvements

### 1. Outliers Manager

### 2. n-Folder Validation


### Application

## III. Second model: Linear Regression

### 1. Prediction Function

### 2. Penalized Loss Function

### 3. Gradient Descent

### Application

## Tests post-fonctions

In [ ]:
sampleN418.isna().sum()
a=managing_missing_val(sampleN418)
a.isna().sum()